<a href="https://colab.research.google.com/github/UniVR-DH/DKR-course/blob/main/L18-advanced/DuckPGQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DuckDB PGQ

An example implementation of the new [SQL/PGQ standard on DuckDB](https://duckpgq.org/documentation/sql_pgq/)

> SQL/PGQ is a graph query language built on top of SQL, designed to bring graph pattern matching capabilities to both seasoned SQL users and those new to graph technology. Standardized by the International Organization for Standardization (ISO), it offers a declarative approach to querying property graphs, which store nodes, edges, and properties.



In [1]:
%pip install duckdb

In [2]:
# Install duckpgq libraries
import duckdb
conn = duckdb.connect(database=':memory:', read_only=False);
conn.install_extension("duckpgq", repository="community")
conn.load_extension("duckpgq")
print("DuckDB and DuckPGQ initialized successfully!")

DuckDB and DuckPGQ initialized successfully!


In [3]:
# Download the data in relational tables
conn.execute("CREATE TABLE Person AS SELECT * FROM 'https://gist.githubusercontent.com/Dtenwolde/2b02aebbed3c9638a06fda8ee0088a36/raw/8c4dc551f7344b12eaff2d1438c9da08649d00ec/person-sf0.003.csv';")
conn.execute("CREATE TABLE Person_knows_person AS SELECT * FROM 'https://gist.githubusercontent.com/Dtenwolde/81c32c9002d4059c2c3073dbca155275/raw/8b440e810a48dcaa08c07086e493ec0e2ec6b3cb/person_knows_person-sf0.003.csv';");

In [4]:
print("First 4 rows of 'Person' table:")
conn.execute("SELECT * FROM Person LIMIT 4;").df()

First 4 rows of 'Person' table:


,creationDate,id,firstName,lastName,gender,birthday,locationIP,browserUsed,LocationCityId,speaks,email
0,2010-01-03 23:10:31.499000+00:00,14,Hossein,Forouhar,male,1984-03-11,77.245.239.11,Firefox,1166,fa;ku;en,Hossein14@hotmail.com
1,2010-01-31 21:13:03.929000+00:00,16,Jan,Zakrzewski,female,1986-07-05,31.41.169.140,Chrome,1284,pl;en,Jan16@hotmail.com;Jan16@gmx.com;Jan16@gmail.co...
2,2010-02-13 06:05:24.513000+00:00,32,Miguel,Gonzalez,male,1981-09-17,148.204.226.31,Chrome,737,es;en,Miguel32@gmx.com;Miguel32@gmail.com;Miguel32@h...
3,2010-03-25 01:14:04.882000+00:00,2199023255557,Eric,Mettacara,male,1989-08-05,203.215.63.48,Firefox,1014,my;en,Eric2199023255557@gmx.com;Eric2199023255557@gm...


In [5]:
print("First 4 rows of 'Person_knows_person' table:")
conn.execute("SELECT * FROM Person_knows_person LIMIT 4;").df()

First 4 rows of 'Person_knows_person' table:


,creationDate,Person1Id,Person2Id
0,2012-10-07 02:24:40.381000+00:00,14,10995116277782
1,2012-07-08 15:27:12.264000+00:00,14,24189255811081
2,2012-11-26 06:45:21.004000+00:00,14,26388279066668
3,2011-11-08 06:05:10.543000+00:00,16,2199023255594


In [6]:
# Declare a PG as a View over the two tables
conn.execute("""
CREATE PROPERTY GRAPH snb
VERTEX TABLES (
    Person
  )
EDGE TABLES (
    Person_knows_person
        SOURCE KEY ( person1id ) REFERENCES Person ( id )
        DESTINATION KEY ( person2id ) REFERENCES Person ( id )
        LABEL Knows
  );
""")

In [7]:
firstName = 'Jan'

result = conn.execute(f"""
FROM GRAPH_TABLE(snb
    MATCH (a:Person WHERE a.firstName = '{firstName}')-[k:Knows]->(b:Person)
    COLUMNS (b.firstName)
);
""")

In [8]:
for row in result.fetchall():
    print(row)

('Ali',)
('Otto',)
('Bryn',)
('Hans',)


In [9]:
firstName = 'Jan'

result = conn.execute(f"""
FROM GRAPH_TABLE (snb
    MATCH p = ANY SHORTEST (a:Person WHERE a.firstName = '{firstName}')-[k:knows]->+(b:Person)
    COLUMNS (path_length(p) AS length, b.firstName)
  )
ORDER BY length
LIMIT 20;
""")

In [10]:
for row in result.fetchall():
    print(row)

(1, 'Bryn')
(1, 'Hans')
(1, 'Otto')
(1, 'Ali')
(2, 'Celso')
(2, 'Alim')
(2, 'Joakim')
(2, 'John')
(2, 'Roberto')
(2, 'Alexei')
(2, 'Evangelos')
(2, 'Mehmet')
(2, 'Jose')
(2, 'Yahya Ould Ahmed El')
(2, 'Ali')
(2, 'Aleksandr')
(2, 'Tissa')
(2, 'Neil')
(2, 'Ashok')
(3, 'Jimmy')
